In [1]:

# ============================================================
# RAILPULSE AI — FEATURE ABLATION EXPERIMENT
# Full 9 Features vs Reduced 4 Features
# ============================================================

import os
import gc
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DATA_PATH = "../data/processed/railpulse_model_features.csv"

MANIFEST_PATH = (
    "../outputs/tables/three_way_train_split_manifest.csv"
)

OUTPUT_DIR = "../outputs/tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100_000
RANDOM_STATE = 42

FULL_FEATURES = [
    "current_arr_delay",
    "distance_to_next",
    "current_distance",
    "current_sequence",
    "arrival_hour_sin",
    "arrival_hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend"
]

REDUCED_FEATURES = [
    "current_arr_delay",
    "distance_to_next",
    "current_distance",
    "current_sequence"
]

TARGET_COL = "next_arr_delay"

# ------------------------------------------------------------
# 2. LOAD INTERNAL SPLIT MANIFEST
# ------------------------------------------------------------

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"train": str}
)

train_ids = set(
    manifest.loc[
        manifest["split"] == "internal_train",
        "train"
    ]
)

validation_ids = set(
    manifest.loc[
        manifest["split"] == "validation",
        "train"
    ]
)

assert train_ids.isdisjoint(validation_ids)

print("=" * 70)
print("RAILPULSE AI — FEATURE ABLATION")
print("=" * 70)

print("Training trains:", len(train_ids))
print("Validation trains:", len(validation_ids))

# ------------------------------------------------------------
# 3. LOAD DATA IN CHUNKS
# ------------------------------------------------------------

USE_COLS = (
    ["train"] + FULL_FEATURES + [TARGET_COL]
)

X_train_parts = []
y_train_parts = []

X_val_parts = []
y_val_parts = []

total_rows = 0

print("\nLoading internal training and validation data...")

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=USE_COLS,
    dtype={"train": str},
    chunksize=CHUNK_SIZE
):

    total_rows += len(chunk)

    train_mask = chunk["train"].isin(
        train_ids
    ).to_numpy()

    val_mask = chunk["train"].isin(
        validation_ids
    ).to_numpy()

    X = chunk[FULL_FEATURES].to_numpy(
        dtype=np.float32
    )

    y = chunk[TARGET_COL].to_numpy(
        dtype=np.float32
    )

    if train_mask.any():
        X_train_parts.append(X[train_mask])
        y_train_parts.append(y[train_mask])

    if val_mask.any():
        X_val_parts.append(X[val_mask])
        y_val_parts.append(y[val_mask])

    if total_rows % 300_000 < CHUNK_SIZE:
        print(f"Processed {total_rows:,} rows")

X_train_full = np.concatenate(X_train_parts)
y_train = np.concatenate(y_train_parts)

X_val_full = np.concatenate(X_val_parts)
y_val = np.concatenate(y_val_parts)

del X_train_parts, y_train_parts
del X_val_parts, y_val_parts
gc.collect()

print("\nDATA LOADED")
print("Training:", X_train_full.shape)
print("Validation:", X_val_full.shape)

# ------------------------------------------------------------
# 4. PREPARE REDUCED FEATURE ARRAYS
# ------------------------------------------------------------

reduced_indices = [
    FULL_FEATURES.index(feature)
    for feature in REDUCED_FEATURES
]

X_train_reduced = X_train_full[
    :, reduced_indices
].copy()

X_val_reduced = X_val_full[
    :, reduced_indices
].copy()

# ------------------------------------------------------------
# 5. DEFINE EVALUATION FUNCTION
# ------------------------------------------------------------

results = []

def evaluate_ablation(
    name,
    features,
    X_train,
    X_validation
):

    delay_index = features.index(
        "current_arr_delay"
    )

    current_train = X_train[:, delay_index]
    current_val = X_validation[:, delay_index]

    # Delta target
    delta_target = (
        y_train - current_train
    ).astype(np.float32)

    model = HistGradientBoostingRegressor(
        loss="absolute_error",
        learning_rate=0.06,
        max_iter=100,
        max_leaf_nodes=31,
        min_samples_leaf=40,
        l2_regularization=2.0,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=12,
        random_state=RANDOM_STATE
    )

    print("\nTraining:", name)

    start = time.time()

    model.fit(
        X_train,
        delta_target
    )

    predicted_change = model.predict(
        X_validation
    )

    predictions = np.maximum(
        current_val + predicted_change,
        0
    )

    elapsed = time.time() - start

    mae = mean_absolute_error(
        y_val,
        predictions
    )

    mse = mean_squared_error(
        y_val,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_val,
        predictions
    )

    results.append({
        "Model": name,
        "Feature_Count": len(features),
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time_Seconds": elapsed
    })

    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R2  :", round(r2, 4))
    print("Time:", round(elapsed, 2), "seconds")

    del model, predictions
    del predicted_change, delta_target
    gc.collect()

# ------------------------------------------------------------
# 6. FULL FEATURE MODEL
# ------------------------------------------------------------

evaluate_ablation(
    "Full Delta Model (9 Features)",
    FULL_FEATURES,
    X_train_full,
    X_val_full
)

# ------------------------------------------------------------
# 7. REDUCED FEATURE MODEL
# ------------------------------------------------------------

evaluate_ablation(
    "Reduced Delta Model (4 Features)",
    REDUCED_FEATURES,
    X_train_reduced,
    X_val_reduced
)

# ------------------------------------------------------------
# 8. PERSISTENCE BASELINE
# ------------------------------------------------------------

delay_index = FULL_FEATURES.index(
    "current_arr_delay"
)

persistence_predictions = X_val_full[
    :, delay_index
]

persistence_mae = mean_absolute_error(
    y_val,
    persistence_predictions
)

persistence_mse = mean_squared_error(
    y_val,
    persistence_predictions
)

results.append({
    "Model": "Persistence Baseline",
    "Feature_Count": 1,
    "MAE": persistence_mae,
    "MSE": persistence_mse,
    "RMSE": np.sqrt(persistence_mse),
    "R2": r2_score(y_val, persistence_predictions),
    "Training_Time_Seconds": 0
})

# ------------------------------------------------------------
# 9. FINAL COMPARISON
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "MAE"
).reset_index(drop=True)

print("\n" + "=" * 75)
print("FEATURE ABLATION RESULTS")
print("=" * 75)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 10. SAVE RESULTS
# ------------------------------------------------------------

save_path = os.path.join(
    OUTPUT_DIR,
    "feature_ablation_results.csv"
)

results_df.to_csv(
    save_path,
    index=False
)

print("\nResults saved:", save_path)

print("\nFEATURE ABLATION COMPLETED.")

# Release arrays
del X_train_full, X_val_full
del X_train_reduced, X_val_reduced
del y_train, y_val
gc.collect()


RAILPULSE AI — FEATURE ABLATION
Training trains: 2012
Validation trains: 504

Loading internal training and validation data...
Processed 300,000 rows
Processed 600,000 rows
Processed 900,000 rows

DATA LOADED
Training: (699293, 9)
Validation: (171336, 9)

Training: Full Delta Model (9 Features)
MAE : 9.1626
RMSE: 31.6465
R2  : 0.8522
Time: 11.14 seconds

Training: Reduced Delta Model (4 Features)
MAE : 9.1466
RMSE: 31.6416
R2  : 0.8523
Time: 7.85 seconds

FEATURE ABLATION RESULTS
                           Model  Feature_Count    MAE       MSE    RMSE     R2  Training_Time_Seconds
Reduced Delta Model (4 Features)              4 9.1466 1001.1910 31.6416 0.8523                 7.8517
   Full Delta Model (9 Features)              9 9.1626 1001.5029 31.6465 0.8522                11.1438
            Persistence Baseline              1 9.2577 1003.8248 31.6832 0.8519                 0.0000

Results saved: ../outputs/tables\feature_ablation_results.csv

FEATURE ABLATION COMPLETED.


0